In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split as tts

In [ ]:
df = pd.read_csv("Advertising.csv")
df.head(3)

,Unnamed: 0,TV,Radio,Newspaper,Sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3


In [ ]:
df = df.drop("Unnamed: 0", axis=1)

In [ ]:
df.head(2)

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4


In [ ]:
x = df.drop("Sales", axis=1)
y = df['Sales']

print(x.shape)
print(y.shape)

(200, 3)
(200,)


In [ ]:
x_train,x_test,y_train,y_test = tts(x,y, test_size=0.2, random_state=42)

In [ ]:
lr = LinearRegression()
lr.fit(x_train,y_train)

LinearRegression()

In [ ]:
print("coeff:",lr.coef_)
print("intercept:",lr.intercept_)

coeff: [0.04472952 0.18919505 0.00276111]
intercept: 2.979067338122629


In [ ]:
y_pred = lr.predict(x_test)

from sklearn.metrics import r2_score

r2 = r2_score(y_test,y_pred)
print(r2)

0.899438024100912


<h1> Using Class

In [ ]:
x_train.shape

(160, 3)

coeff: [0.04472952 0.18919505 0.00276111]
intercept: 2.979067338122629

In [ ]:
class GDRegressor:

  def __init__(self,learning_rate=0.01,epochs=100):
    self.learning_rate = learning_rate
    self.epochs = epochs
    self.coef_ = None
    self.intercept_ = None

  def fit(self,x_train,y_train):

    # initializing random coef & intercept
    self.coef_ = np.ones(x_train.shape[1])
    self.intercept_ = 0

    print(self.coef_,self.intercept_)

    for i in range(self.epochs):

        y_hat = np.dot(x_train, self.coef_) + self.intercept_

        intercept_der = -2 * np.mean(y_train - y_hat)
        self.intercept_ -= self.learning_rate * intercept_der

        coef_der = -2 * np.dot((y_train - y_hat), x_train) / x_train.shape[0]
        self.coef_ -= self.learning_rate * coef_der

        # print control
        if i < 200 or i >= self.epochs - 200:
            print(f"epoch={i}, coef={self.coef_}, intercept={self.intercept_}")


  def predict(self,x_test):
    return np.dot(x_test,self.coef_) + self.intercept_

In [ ]:
gd = GDRegressor(epochs=90000,learning_rate=0.00003)

In [ ]:
gd.fit(x_train,y_train)

[1. 1. 1.] 0
epoch=0, coef=[-1.11000196  0.71956697  0.62877668], intercept=-0.0113244375
epoch=1, coef=[0.68557469 0.91239919 0.87304452], intercept=-0.00260382932026167
epoch=2, coef=[-0.81335674  0.70797238  0.60136423], intercept=-0.010749576220267175
epoch=3, coef=[0.46546611 0.84036473 0.76780039], intercept=-0.004633987341707923
epoch=4, coef=[-0.59899984  0.69040839  0.56760341], intercept=-0.01051041325073479
epoch=5, coef=[0.31213281 0.78020606 0.67933472], intercept=-0.006239533416515967
epoch=6, coef=[-0.44346389  0.6693674   0.53060352], intercept=-0.010494165441627148
epoch=7, coef=[0.20601287 0.7292288  0.6040215 ], intercept=-0.007527932334199647
epoch=8, coef=[-0.3300321   0.64655581  0.4924638 ], intercept=-0.010621997944943926
epoch=9, coef=[0.13321743 0.68545324 0.5391745 ], intercept=-0.008576938329001287
epoch=10, coef=[-0.24679127  0.6231267   0.45457492], intercept=-0.010839100724692005
epoch=11, coef=[0.08389043 0.64741344 0.4827853 ], intercept=-0.009443058626

In [ ]:
y_pred = gd.predict(x_test)

In [ ]:
r2 = r2_score(y_test, y_pred)

In [ ]:
r2

0.8904600352844214

<h1> Scaled values

In [ ]:
from math import inf
x_scaled = (x - x.mean()) / x.std()
x_train,x_test,y_train,y_test = tts(x_scaled,y, test_size=0.2, random_state=42)



class GDRegressor:

  def __init__(self,learning_rate=0.01,epochs=100, tolerance = 1e-17, patience = 15):
    self.learning_rate = learning_rate
    self.epochs = epochs
    self.coef_ = None
    self.intercept_ = None
    self.tolerance = tolerance
    self.patience = patience

  def fit(self,x_train,y_train):

    # initializing random coef & intercept
    self.coef_ = np.ones(x_train.shape[1])
    self.intercept_ = 0

    previous_loss = float(inf)
    patience_counter = 0

    print(self.coef_,self.intercept_)

    for i in range(self.epochs):

        y_hat = np.dot(x_train, self.coef_) + self.intercept_

        # Loss (MSE)
        loss = np.mean((y_train - y_hat) ** 2)

        intercept_der = -2 * np.mean(y_train - y_hat)
        self.intercept_ -= self.learning_rate * intercept_der

        coef_der = -2 * np.dot((y_train - y_hat), x_train) / x_train.shape[0]
        self.coef_ -= self.learning_rate * coef_der

        # print control
        if i < 200 or i >= self.epochs - 200:
            print(f"epoch={i}, coef={self.coef_}, intercept={self.intercept_}")


        # early stopping
        if abs(previous_loss - loss) < self.tolerance:
          patience_counter +=1
          if patience_counter >= self.patience:
            print(f"Early stopping at epochs {i}")
            break
        else:
          patience_counter = 0
        previous_loss = loss

  def predict(self,x_test):
    return np.dot(x_test,self.coef_) + self.intercept_

In [ ]:
x_train.head(2)

,TV,Radio,Newspaper
79,-0.361572,-1.048306,-0.342262
197,0.348934,-0.940539,-1.109069


In [ ]:
gd = GDRegressor(epochs=100,learning_rate=0.3)

In [ ]:
gd.fit(x_train,y_train)

[1. 1. 1.] 0
epoch=0, coef=[2.97822504 1.73788918 0.69414594], intercept=8.471651837977642
epoch=1, coef=[3.6185502  2.17574013 0.51060985], intercept=11.825612325860828
epoch=2, coef=[3.80776149 2.42625105 0.3779445 ], intercept=13.15767110037567
epoch=3, coef=[3.85360274 2.57182079 0.28068771], intercept=13.688266506104574
epoch=4, coef=[3.85839977 2.65892715 0.21105239], intercept=13.900205647207935
epoch=5, coef=[3.85400972 2.71256341 0.16235157], intercept=13.985081528335819
epoch=6, coef=[3.84920823 2.7463585  0.12887883], intercept=14.019148788850643
epoch=7, coef=[3.84575367 2.7680122  0.10614617], intercept=14.032844981153136
epoch=8, coef=[3.84356223 2.78204811 0.09083142], intercept=14.038354173101096
epoch=9, coef=[3.84223775 2.79121691 0.08056955], intercept=14.04056705260523
epoch=10, coef=[3.84144918 2.79723691 0.07371823], intercept=14.041451670174125
epoch=11, coef=[3.84097864 2.80120259 0.06915499], intercept=14.041801575114755
epoch=12, coef=[3.8406947  2.80382059 0.

In [ ]:
y_pred = gd.predict(x_test)

In [ ]:
r2 = r2_score(y_test, y_pred)
r2

0.899438024100912

In [ ]:
sk_pred = 0.899438024100912
gd_pred = r2

if gd_pred > sk_pred:
  diff_gd = gd_pred - sk_pred
  print("GD working better:",diff_gd)
elif gd_pred < sk_pred:
  diff_sk = sk_pred - gd_pred
  print("sklearn model working better with",diff_sk)
else:
  print("No difference both got same score")

No difference both got same score
